In [11]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
SUMMARY_DIR = Path("./results")
files = sorted(SUMMARY_DIR.glob("summary_*.csv"))

df = pd.concat(
    (pd.read_csv(f, parse_dates=["day"]) for f in files),
    ignore_index=True
)


SUMMARY_DIR = Path("./results_open_positions")
files = sorted(SUMMARY_DIR.glob("summary_*.csv"))

df_open_pos = pd.concat(
    (pd.read_csv(f, parse_dates=["day"]) for f in files),
    ignore_index=True
)


df["day"] = pd.to_datetime(df["day"] , utc=True).dt.normalize()

df_open_pos["day"] = pd.to_datetime(df_open_pos["day"] , utc=True).dt.normalize()


ri_base = (
    df[df["combo_id"] == 0][["day", "profit"]]
    .rename(columns={"profit": "RI_base"})
)

df = df.merge(ri_base, on="day", how="left")

df["uplift_abs"] = df["profit"] - df["RI_base"]
df["uplift_pct"] = df["uplift_abs"] / df["RI_base"]



df_open_pos = df_open_pos.merge(ri_base, on="day", how="left")

df_open_pos["uplift_abs"] = df_open_pos["profit"] - df_open_pos["RI_base"]
df_open_pos["uplift_pct"] = df_open_pos["uplift_abs"] / df_open_pos["RI_base"]

In [6]:
#df = df.loc[df["day"] < pd.Timestamp("2020-12-31", tz="UTC")].copy()
#df_open_pos = df_open_pos.loc[df_open_pos["day"] < pd.Timestamp("2020-12-31", tz="UTC")].copy()

df.loc[(df["kind"] != "no_da") & (df["sell_hour"].isna()), "sell_hour"] = 0

#### Load drl observation space features

In [7]:
drl_obs_data = pd.read_csv("../data/simplified_data_jan_with_exaa_and_id_full/df_spot_train_2019-01-01_2024-12-31_with_features_utc.csv")
drl_obs_data["time"] = pd.to_datetime(drl_obs_data["time"], utc=True)
drl_obs_data.set_index("time", inplace=True)
drl_obs_data = drl_obs_data.resample('D').mean()
drl_obs_data.reset_index(inplace=True)
drl_obs_data["date"] = drl_obs_data["time"].dt.date
drl_obs_data

df["day"] = pd.to_datetime(df["day"].astype(str).str.slice(0, 10), errors="coerce")
df["day_utc"] = df["day"].dt.tz_localize("UTC")
df["date"] = df["day_utc"].dt.date
df

df_merged = df.merge(
    drl_obs_data,
    on="date",
    how="left"
)

In [8]:
df1= df_merged.copy()

In [9]:
columns_to_keep = [ 'date',"combo_id", 'buy_hour', 'sell_hour','profit',
      'uplift_abs', 'uplift_pct',
      'da_profit', 'da_buy_price', 'da_sell_price',
       'load_forecast_d_minus_1_1000_total_de_lu_mw',
       'pv_forecast_d_minus_1_1000_de_lu_mw',
       'wind_offshore_forecast_d_minus_1_1000_de_lu_mw',
       'wind_onshore_forecast_d_minus_1_1000_de_lu_mw', 'date_month',
       'day_of_week', 'wind_forecast_daily_mean', 'wind_forecast_daily_std',
       'spread_id_full_da_h_mean', 'spread_id_full_da_h_std',
       'spread_id_full_da_h_min', 'spread_id_full_da_h_max',
       'spread_id_full_da_qh_mean', 'spread_id_full_da_qh_std',
       'spread_id_full_da_qh_min', 'spread_id_full_da_qh_max',
       'exaa_pf_daily_mean', 'exaa_pf_daily_std', 'exaa_pf_daily_min',
       'exaa_pf_daily_max', 'exaa_pf_daily_spread', 'exaa_pf_daily_diff_sum',
       'exaa_pf_daily_diff_max']

df1= df_merged[columns_to_keep]



df1["buy_hour"] = df1["buy_hour"].fillna(0)
df1["sell_hour"] = df1["sell_hour"].fillna(0)

/var/folders/83/hxkk8qrs62xfzh1l_vbb4grc0000gn/T/ipykernel_20267/2617145396.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["buy_hour"] = df1["buy_hour"].fillna(0)
/var/folders/83/hxkk8qrs62xfzh1l_vbb4grc0000gn/T/ipykernel_20267/2617145396.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["sell_hour"] = df1["sell_hour"].fillna(0)


In [13]:
df1.columns

Index(['date', 'combo_id', 'buy_hour', 'sell_hour', 'profit', 'uplift_abs',
       'uplift_pct', 'da_profit', 'da_buy_price', 'da_sell_price',
       'load_forecast_d_minus_1_1000_total_de_lu_mw',
       'pv_forecast_d_minus_1_1000_de_lu_mw',
       'wind_offshore_forecast_d_minus_1_1000_de_lu_mw',
       'wind_onshore_forecast_d_minus_1_1000_de_lu_mw', 'date_month',
       'day_of_week', 'wind_forecast_daily_mean', 'wind_forecast_daily_std',
       'spread_id_full_da_h_mean', 'spread_id_full_da_h_std',
       'spread_id_full_da_h_min', 'spread_id_full_da_h_max',
       'spread_id_full_da_qh_mean', 'spread_id_full_da_qh_std',
       'spread_id_full_da_qh_min', 'spread_id_full_da_qh_max',
       'exaa_pf_daily_mean', 'exaa_pf_daily_std', 'exaa_pf_daily_min',
       'exaa_pf_daily_max', 'exaa_pf_daily_spread', 'exaa_pf_daily_diff_sum',
       'exaa_pf_daily_diff_max'],
      dtype='object')

In [12]:
# Target
y_col = "profit"

# Groups (pro Tag)
group_col = "date"
groups = df[group_col]

# Excludes (Targets + Zeitspalten etc.)
exclude = {
    "uplift_abs", "uplift_pct",
    "uplift_pos", "uplift_pos_5pct",
    "day", "time", "datetime", group_col
}
feature_cols = [c for c in df.columns if c not in exclude]

X = df1[feature_cols]
y = df1[y_col]

# Split by day
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx]



use_lgbm = True
try:
    from lightgbm import LGBMRegressor
except Exception:
    use_lgbm = False

if use_lgbm:
    model = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        max_depth= 10,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )
else:
    from sklearn.ensemble import HistGradientBoostingRegressor
    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=None,
        max_iter=800,
        random_state=42
    )

model.fit(X_train, y_train)

pred = model.predict(X_test)

rmse = mean_squared_error(y_test, pred)
mae  = mean_absolute_error(y_test, pred)
r2   = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)


KeyError: "['kind', 'da_exec_time', 'da_profit_share', 'RI_base', 'day_utc'] not in index"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- direkt nach pred = model.predict(X_test) ---
plot_df = df1.iloc[test_idx].copy()
plot_df["profit_actual"] = y_test.values
plot_df["profit_forecast"] = pred

# (optional) falls day nicht datetime ist: sauber casten
plot_df["day"] = pd.to_datetime(plot_df["day"]).dt.date
plot_df["combo_id"] = plot_df["combo_id"].astype(int)
# Sicherstellen:
plot_df["day"] = pd.to_datetime(plot_df["day"]).dt.date
plot_df["combo_id"] = plot_df["combo_id"].astype(int)

# Baseline pro Tag aus actual bei combo_id==0
baseline_map = (
    plot_df.loc[plot_df["combo_id"] == 0]
           .groupby("day")["profit_actual"]
           .mean()  # mean falls es doch Duplikate gibt
)

plot_df["baseline"] = plot_df["day"].map(baseline_map)


# Optional: Hinweis, falls für manche Tage combo_id==0 fehlt
missing_days = plot_df.loc[plot_df["baseline"].isna(), "day"].unique()
if len(missing_days) > 0:
    print(f"Warnung: Für {len(missing_days)} Tag(e) fehlt combo_id==0 im Testset, baseline ist dort NaN.")


from matplotlib.ticker import MultipleLocator
def plot_random_day_with_baseline(plot_df, seed=None, day=None):
    rng = np.random.default_rng(seed)

    days = plot_df["day"].dropna().unique()
    if len(days) == 0:
        raise ValueError("Keine 'day'-Werte gefunden.")

    if day is None:
        day = rng.choice(days)

    tmp = plot_df[plot_df["day"] == day].copy().sort_values("combo_id")

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(tmp["combo_id"], tmp["profit_actual"], label="actual", marker="o", linewidth=1)
    ax.plot(tmp["combo_id"], tmp["profit_forecast"], label="forecast", marker="o", linewidth=1)

    for x in range(0, 300, 10):
        ax.axvline(x, linewidth=0.1, alpha=0.8)

    # Baseline als horizontale Linie (konstant pro Tag)
    base = tmp["baseline"].iloc[0] if len(tmp) else np.nan
    if pd.notna(base):
        ax.axhline(base, label="baseline (combo_id=0 actual)", linewidth=2)
    else:
        ax.text(0.01, 0.95, "baseline fehlt (combo_id=0 nicht vorhanden)",
                transform=ax.transAxes, va="top")

    ax.set_title(f"Actual vs Forecast + Baseline — day={day} (n={len(tmp)})")
    ax.set_xlabel("combo_id (0..299)")
    ax.set_ylabel("profit")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    return day